# UC13 Pipeline — End-to-End Test Notebook

Runs each production script's `main()` in sequence, exactly as the Databricks Workflow does.
Intermediate verification queries between steps let you inspect state before continuing.

**Execution order:**
```
[ 1. Config ]  →  [ 2. Setup VS ]  →  [ 3. Download/Upload ]
  →  [ 4. Verify upload_log ]  →  [ 5. Classifier ]
  →  [ 6. Verify doc_relevance ]  →  [ 7. Parser ]
  →  [ 8. Verify chunks/embeddings ]  →  [ 9. Profiler ]
  →  [ 10. Verify profile ]

Phase 3 agents (run after Profiler):
  →  [ 11. Business Model Agent ]  →  [ 12. Financial Trends Agent ]
  →  [ 12b. Financial Story Assessment (markdown report) ]
  →  [ 13. Customer Quality Agent ] →  [ 14. KPI Agent ]
  →  [ 15. Legal Contracts Agent ]  →  [ 16. Quality of Earnings Agent ]
  →  [ 17. Verify all analysis tables ]
```

**Before running:**
- Confirm the repo is cloned under `/Workspace/Repos/`
- Confirm secrets scope `uc13` is populated (see `workflows/README.md`)
- Set `SP_COMPANY_NAME` in the widget below

In [ ]:
# ── Cell 0: Install dependencies ──────────────────────────────────────────────
# Run this cell ONCE per cluster session (or after a cluster restart).
# restartPython() forces a clean kernel — always re-run Cell 1 (Config) after this.

%pip install -q msal>=1.28.0 requests>=2.31.0 "python-docx>=1.1.0" "openpyxl>=3.1.0" "typing_extensions>=4.6.0" mlflow>=2.13.0

dbutils.library.restartPython()

In [ ]:
# ── Cell 1: Config ────────────────────────────────────────────────────────────
# Edit SP_COMPANY_NAME to switch companies. Everything else uses safe defaults.

import os, sys
from pathlib import Path

# Widgets so the Workflow UI can also inject values.
dbutils.widgets.text("sp_company_name",     "Elder Care")
dbutils.widgets.text("catalog",             "uc13")
dbutils.widgets.text("schema",              "ingestion")
dbutils.widgets.text("embedding_endpoint",  "databricks-bge-large-en")
dbutils.widgets.text("llm_endpoint",        "databricks-meta-llama-3-3-70b-instruct")
dbutils.widgets.dropdown("parse_priority_tiers", "all", ["all", "1", "2", "3", "1,2", "1,2,3"])

# Mirror widget values into os.environ so get_param() in imported modules can
# read them reliably via the os.environ fallback (dbutils.widgets.get() can
# silently fail when called from inside an imported Python module).
os.environ["sp_company_name"]    = dbutils.widgets.get("sp_company_name")
os.environ["catalog"]            = dbutils.widgets.get("catalog")
os.environ["schema"]             = dbutils.widgets.get("schema")
os.environ["embedding_endpoint"] = dbutils.widgets.get("embedding_endpoint")
os.environ["llm_endpoint"]       = dbutils.widgets.get("llm_endpoint")
os.environ["parse_priority_tiers"] = dbutils.widgets.get("parse_priority_tiers")

# ── Resolve repo root from this notebook's path ───────────────────────────────
def get_current_path():
    try:
        nb_path = (
            dbutils.notebook.entry_point
            .getDbutils().notebook().getContext()
            .notebookPath().get()
        )
        return Path("/Workspace") / nb_path.lstrip("/")
    except Exception:
        return Path(os.getcwd())

def find_repo_root(marker="agents"):
    p = get_current_path()
    if p.is_file():
        p = p.parent
    for candidate in [p, *p.parents]:
        if (candidate / marker).exists():
            return str(candidate)
    raise RuntimeError(f"Could not find a parent directory containing '{marker}'")

REPO_ROOT    = find_repo_root()
SCRIPTS      = str(Path(REPO_ROOT) / "jobs" / "scripts")
WORKSTREAMS  = str(Path(REPO_ROOT) / "agents" / "workstreams")

for p in [REPO_ROOT, SCRIPTS, WORKSTREAMS]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"repo_root   : {REPO_ROOT}")
print(f"scripts     : {SCRIPTS}")
print(f"workstreams : {WORKSTREAMS}")
print(f"company     : {dbutils.widgets.get('sp_company_name')}")
print(f"parse tiers : {dbutils.widgets.get('parse_priority_tiers')}")


In [ ]:
# ── Cell 1b: Discover & Select Company ────────────────────────────────────────
# Fetches the available company folders from SharePoint and turns sp_company_name
# into a dropdown so you can pick without typing.
#
# Workflow:
#   1. Run this cell → dropdown populates with live companies from SharePoint.
#   2. Select your company from the dropdown.
#   3. Re-run this cell → selection is confirmed and synced to os.environ.

import importlib
import sys
from pathlib import Path

# Ensure agents package is on the path (REPO_ROOT set by Cell 1).
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from agents.ingestion.tools import connector as _connector
importlib.reload(_connector)

companies = _connector.list_companies()

print("Available companies in SharePoint:")
for i, name in enumerate(companies, 1):
    print(f"  {i}. {name}")

# Replace the text widget with a live dropdown.
try:
    dbutils.widgets.remove("sp_company_name")
except Exception:
    pass

default = companies[0] if companies else "Elder Care"
dbutils.widgets.dropdown("sp_company_name", default, companies or ["Elder Care"])

# Sync the selected value into os.environ so imported scripts can read it.
selected = dbutils.widgets.get("sp_company_name")
os.environ["sp_company_name"] = selected

print(f"\n✓ Selected: '{selected}'")
print("  Change the dropdown above and re-run this cell to switch companies.")

---
## Step 0 — Setup Vector Search
Run this cell **once** per workspace. Skip it on subsequent runs — it is idempotent but slow.

In [ ]:
# ── Cell 2: Setup Vector Search (run once) ────────────────────────────────────
# Comment out after the first successful run.

import importlib, setup_vector_search as s0
importlib.reload(s0)
s0.main()

---
## Step 1 — Download & Upload (Phase 1)
Downloads from SharePoint → local scratch → UC Volume.  
Writes `uc13.ingestion.upload_log` with Priority Tier signals.

In [ ]:
# ── Cell 3: Download & Upload ─────────────────────────────────────────────────
import importlib, download_upload as s1
importlib.reload(s1)
s1.main()

In [ ]:
# ── Cell 4: Verify upload_log ─────────────────────────────────────────────────
catalog      = dbutils.widgets.get("catalog")
schema       = dbutils.widgets.get("schema")
company_name = dbutils.widgets.get("sp_company_name")

display(
    spark.sql(f"""
        SELECT
            priority_tier,
            format,
            COUNT(*) AS files,
            SUM(size_bytes) / 1e6 AS total_mb
        FROM {catalog}.{schema}.upload_log
        WHERE company_name = '{company_name}'
        GROUP BY priority_tier, format
        ORDER BY priority_tier ASC NULLS LAST, files DESC
    """)
)

# Spot-check: which files got Priority Tier 1?
display(
    spark.sql(f"""
        SELECT file_name, folder_path, priority_reason, mod_date
        FROM {catalog}.{schema}.upload_log
        WHERE company_name = '{company_name}'
          AND priority_tier = 1
        ORDER BY folder_path
    """)
)

---
## Step 2 — Document Classifier (Phase 2a)
Calls the LLM in batches of 20 to assign workstream tags + priority tier.  
Writes `uc13.classification.doc_relevance`.

In [ ]:
# ── Cell 5: Document Classifier ───────────────────────────────────────────────
import importlib, document_classifier as s2
importlib.reload(s2)
s2.main()

In [ ]:
# ── Cell 6: Verify doc_relevance ──────────────────────────────────────────────
company_name = dbutils.widgets.get("sp_company_name")

# Distribution by workstream tag and tier
display(
    spark.sql(f"""
        SELECT tag, priority_tier, COUNT(*) AS files
        FROM (
            SELECT explode(workstream) AS tag, priority_tier
            FROM uc13.classification.doc_relevance
            WHERE company_name = '{company_name}'
        ) t
        GROUP BY tag, priority_tier
        ORDER BY priority_tier ASC NULLS LAST, files DESC
    """)
)

# Files flagged for parsing, broken down by tier
display(
    spark.sql(f"""
        SELECT
            should_parse,
            priority_tier,
            extraction_confidence,
            COUNT(*) AS files
        FROM uc13.classification.doc_relevance
        WHERE company_name = '{company_name}'
        GROUP BY should_parse, priority_tier, extraction_confidence
        ORDER BY should_parse DESC, priority_tier ASC NULLS LAST
    """)
)

# Spot-check: verify Tier 1 files look correct
display(
    spark.sql(f"""
        SELECT filename, folder_path, workstream, priority_reason, extraction_confidence
        FROM uc13.classification.doc_relevance
        WHERE company_name = '{company_name}'
          AND priority_tier = 1
        ORDER BY folder_path
    """)
)

# Red flag: should_parse=true with null/empty workstream
suspicious = spark.sql(f"""
    SELECT filename, folder_path, workstream
    FROM uc13.classification.doc_relevance
    WHERE company_name = '{company_name}'
      AND should_parse = true
      AND (workstream IS NULL OR size(workstream) = 0)
""").count()
print(f"\n⚠️  Files with should_parse=true but no workstream tag: {suspicious}")

---
## Step 3 — Ingestion Parser (Phase 2b)
Parses `should_parse=true` files into chunks, generates embeddings.  
Priority Tier files are processed first.  
Writes `uc13.ingestion.chunks` and `uc13.ingestion.embeddings`.

In [ ]:
# ── Cell 7: Ingestion Parser ──────────────────────────────────────────────────
import importlib, ingestion_parser as s3
importlib.reload(s3)
s3.main()

In [ ]:
# ── Cell 8: Verify chunks + embeddings ────────────────────────────────────────
company_name = dbutils.widgets.get("sp_company_name")

# Chunk stats by file type
display(
    spark.sql(f"""
        SELECT
            file_type,
            COUNT(DISTINCT doc_id)  AS documents,
            COUNT(*)                AS total_chunks,
            ROUND(AVG(char_count))  AS avg_chars,
            MIN(char_count)         AS min_chars,
            MAX(char_count)         AS max_chars
        FROM uc13.ingestion.chunks
        WHERE company_name = '{company_name}'
        GROUP BY file_type
        ORDER BY file_type
    """)
)

# Embeddings: confirm workstream + priority_tier columns are populated
display(
    spark.sql(f"""
        SELECT
            workstream,
            priority_tier,
            COUNT(*) AS embeddings
        FROM uc13.ingestion.embeddings
        WHERE company_name = '{company_name}'
        GROUP BY workstream, priority_tier
        ORDER BY priority_tier ASC NULLS LAST
    """)
)

# Red flag: chunks with no section_header (carry-forward fix should eliminate these)
headerless = spark.sql(f"""
    SELECT file_name, chunk_index, LEFT(chunk_text, 120) AS preview
    FROM uc13.ingestion.chunks
    WHERE company_name = '{company_name}'
      AND section_header IS NULL
    LIMIT 10
""").count()
print(f"\n⚠️  Chunks with null section_header: {headerless}")

# Red flag: oversized Excel chunks that could truncate in BGE
display(
    spark.sql(f"""
        SELECT file_name, section_header, char_count
        FROM uc13.ingestion.chunks
        WHERE company_name = '{company_name}'
          AND file_type = 'xlsx' AND char_count > 32000
        ORDER BY char_count DESC
        LIMIT 10
    """)
)

In [ ]:
# ── Cell 8b: Smoke-test semantic search ───────────────────────────────────────
# Quick retrieval check before running the profiler.

from agents.shared.retrieval import semantic_search

results = semantic_search(
    query="company overview what does this company do business description",
    spark=spark,
    top_k=5,
    file_name_filter=["CIM", "Business", "Overview"],
    workstream_filter=["BUSINESS_MODEL"],
    min_chunk_length=100,
)

print(f"\nReturned {len(results)} chunks\n")
for r in results:
    print(f"  [{r.file_name}]  section: {r.section_header}")
    print(f"  {r.chunk_text[:200]}")
    print()

---
## Step 4 — Company Profiler (Phase 2b)
Retrieves context for each profiling dimension, calls the LLM, and saves a structured profile.  
`company_name` always comes from the widget — never from LLM output.

In [ ]:
# ── Cell 9: Company Profiler ──────────────────────────────────────────────────
import importlib, company_profiler as s4
importlib.reload(s4)
s4.main()

In [ ]:
# ── Cell 10: Verify company_profile ───────────────────────────────────────────
company_name = dbutils.widgets.get("sp_company_name")

display(
    spark.sql("""
        SELECT
            company_name,
            industry_overlay,
            overlay_confidence,
            revenue_model,
            deal_type,
            banked,
            vertical_subsector,
            data_room_gaps,
            created_at
        FROM uc13.classification.company_profile
        ORDER BY created_at DESC
        LIMIT 5
    """)
)

row = spark.sql(f"""
    SELECT * FROM uc13.classification.company_profile
    WHERE company_name = '{company_name}'
    ORDER BY created_at DESC LIMIT 1
""").collect()[0]

print("\n=== Latest profile ===")
print(f"  company_name          : {row.company_name}")
print(f"  industry_overlay      : {row.industry_overlay} ({row.overlay_confidence})")
print(f"  revenue_model         : {row.revenue_model}")
print(f"  revenue_model_note    : {row.revenue_model_note}")
print(f"  business_description  : {row.business_description}")
print(f"  company_size_indicators: {row.company_size_indicators}")
print(f"  deal_type             : {row.deal_type}")
print(f"  banked                : {row.banked}")
print(f"  banked_note           : {row.banked_note}")
print(f"  vertical_subsector    : {row.vertical_subsector}")
if row.data_room_gaps:
    print(f"  data_room_gaps ({len(row.data_room_gaps)})  :")
    for gap in row.data_room_gaps:
        print(f"    ! {gap}")
else:
    print("  data_room_gaps        : none")

---
## Utilities — Reset / Re-run a single step
Use these cells to truncate a table and re-run a specific step without starting over.

In [ ]:
# ── Cell 11: Reset helpers (run individually as needed) ───────────────────────
# Deletes only the current company's rows — other companies are preserved.
# Uncomment the table(s) you want to reset, run the cell, then re-run the step.

company_name = dbutils.widgets.get("sp_company_name")

# spark.sql(f"DELETE FROM uc13.ingestion.upload_log           WHERE company_name = '{company_name}'")
# spark.sql(f"DELETE FROM uc13.classification.doc_relevance   WHERE company_name = '{company_name}'")
# spark.sql(f"DELETE FROM uc13.ingestion.chunks               WHERE company_name = '{company_name}'")
# spark.sql(f"DELETE FROM uc13.ingestion.embeddings           WHERE company_name = '{company_name}'")
# spark.sql(f"DELETE FROM uc13.classification.company_profile WHERE company_name = '{company_name}'")

print(f"Uncomment the table(s) you want to reset for company='{company_name}', then run this cell.")

In [ ]:
# ── Cell 12: Full pipeline state summary ──────────────────────────────────────
# Shows total rows per table and rows for the current company widget value.

company_name = dbutils.widgets.get("sp_company_name")

tables = {
    "upload_log":      ("uc13.ingestion.upload_log",            "company_name"),
    "doc_relevance":   ("uc13.classification.doc_relevance",    "company_name"),
    "chunks":          ("uc13.ingestion.chunks",                "company_name"),
    "embeddings":      ("uc13.ingestion.embeddings",            "company_name"),
    "company_profile": ("uc13.classification.company_profile",  "company_name"),
}

print(f"{'Table':<25} {'Total':>8}  {'This company':>14}")
print("-" * 52)
for label, (fqt, col) in tables.items():
    try:
        total = spark.sql(f"SELECT COUNT(*) AS n FROM {fqt}").collect()[0]["n"]
        mine  = spark.sql(f"SELECT COUNT(*) AS n FROM {fqt} WHERE {col} = '{company_name}'").collect()[0]["n"]
        print(f"  {label:<23} {total:>8,}  {mine:>14,}")
    except Exception as e:
        print(f"  {label:<23} {'(not found)':>8}")

---
## Phase 3 Pre-flight — Coverage Check & Incremental Ingestion

Run these two cells **before** the Phase 3 agents whenever you see agents returning
0 chunks for a workstream. The most common cause: `ingestion_parser.py` was run with
`parse_priority_tiers=1` and CUSTOMER / QUALITY_EARNINGS documents only appear in
Tier 2.

**`ensure_coverage.py`** is a standalone utility that:
1. Compares `doc_relevance` (what *should* be indexed) against `embeddings` (what *is* indexed).
2. Identifies files approved for the target tiers that have not yet been embedded.
3. Parses and embeds only those missing files — **APPEND only, no DELETE** — so
   already-ingested Tier 1 data is never overwritten.
4. Triggers a Vector Search index sync to make the new embeddings searchable immediately.

> **Rule:** Run Cell 8c first (read-only diagnostic), then Cell 8d if any workstream shows
> "NO COVERAGE". Re-running Cell 8d is safe — it skips files already in `embeddings`.

In [ ]:
# ── Cell 8c: Agent Coverage Diagnostic (read-only) ────────────────────────────
# Shows which workstreams have zero ingested files for the target company.
# No data is written. Run this before Cell 8d to decide whether ingest is needed.
import importlib, sys
from pathlib import Path

sys.path.insert(0, SCRIPTS)   # SCRIPTS = jobs/scripts, set in Cell 1
import ensure_coverage as ec
importlib.reload(ec)

company_name = dbutils.widgets.get("sp_company_name")
catalog      = dbutils.widgets.get("catalog")

# Check Tiers 1 and 2 — the tiers where all workstream-relevant docs should appear.
# Adjust the list if you need to inspect Tier 3 as well: tiers=[1, 2, 3]
report = ec.get_coverage_report(
    company_name=company_name,
    catalog=catalog,
    tiers=[1, 2],
    spark=spark,
)
ec.print_coverage_report(report)

# Surface any workstream that will cause a Phase 3 agent to return 0 chunks.
zero_coverage = [
    ws for ws, counts in report["by_workstream"].items()
    if len(counts["ingested"]) == 0
]
if zero_coverage:
    print(f"⚠  Workstreams with NO ingested files: {zero_coverage}")
    print("   → Run Cell 8d to ingest missing files before running Phase 3 agents.")
else:
    print("✓  All workstreams have at least one ingested file — Phase 3 agents are ready.")

In [ ]:
# ── Cell 8d: Ingest Missing Files (Incremental — APPEND only) ─────────────────
# Parses and embeds only files not yet in uc13.ingestion.embeddings.
# SAFE TO RE-RUN: already-ingested files are detected and skipped automatically.
# Triggers Vector Search index sync on completion.
#
# Run Cell 8c first to see the coverage gap before writing anything.
import importlib

import ensure_coverage as ec
importlib.reload(ec)

company_name       = dbutils.widgets.get("sp_company_name")
catalog            = dbutils.widgets.get("catalog")
embedding_endpoint = dbutils.widgets.get("embedding_endpoint")

# Ingest Tiers 1 and 2.  Change tiers=[1, 2, 3] to also fill Tier 3 gaps.
ingest_summary = ec.ingest_missing(
    company_name=company_name,
    catalog=catalog,
    tiers=[1, 2],
    spark=spark,
    embedding_endpoint=embedding_endpoint,
)

print(f"\n=== Ingest Summary ===")
print(f"  Files processed    : {ingest_summary['files_processed']}")
print(f"  Chunks written     : {ingest_summary['chunks_written']}")
print(f"  Embeddings written : {ingest_summary['embeddings_written']}")
print(f"  Files skipped      : {ingest_summary['skipped']}")
print()

# Re-run coverage check to confirm all workstreams are now covered.
post_report = ec.get_coverage_report(
    company_name=company_name,
    catalog=catalog,
    tiers=[1, 2],
    spark=spark,
)
print("=== Post-Ingest Coverage ===")
ec.print_coverage_report(post_report)

zero_after = [
    ws for ws, counts in post_report["by_workstream"].items()
    if len(counts["ingested"]) == 0
]
if zero_after:
    print(f"⚠  Still no coverage for: {zero_after}")
    print("   Possible causes: files not in the volume, unsupported format, or not classified by this workstream.")
else:
    print("✓  All workstreams covered — ready to run Phase 3 agents.")

In [ ]:
# ── Cell 11a: Business Model table — one-time schema migration ────────────────
# Run this cell ONCE whenever business_model_agent.py has a schema-breaking change
# (new columns, renamed columns, changed types).  It drops the old table so Cell 11
# recreates it with the current schema on the next run.
#
# Safe to re-run: if the table already has the new schema this cell prints a
# confirmation and makes no changes.  All existing rows are lost on a migration —
# intentional; re-run Cell 11 to repopulate.

catalog      = dbutils.widgets.get("catalog")
company_name = dbutils.widgets.get("sp_company_name")
table        = f"{catalog}.analysis.business_model"

_EXPECTED_COLS = {
    "company_name", "cim_detected", "executive_summary",
    "revenue_model_tag", "revenue_model_pct_split", "revenue_model_note",
    "revenue_durability_flag", "flag_confidence", "flag_rule_applied",
    "products_services_json", "customer_profile_json",
    "sales_motion_tag", "sales_motion_json", "revenue_visibility_json",
    "key_dependencies_json", "recent_model_changes_json",
    "overlay_conflict", "overlay_conflict_note", "overlay_conflict_evidence",
    "data_room_gaps", "citations", "reasoning_trace",
    "flags", "report_path", "created_at",
}

try:
    live_cols = {f.name for f in spark.table(table).schema.fields}
    missing   = _EXPECTED_COLS - live_cols
    extra     = live_cols - _EXPECTED_COLS
    if missing:
        print(f"⚠  Schema mismatch detected on {table}")
        print(f"   Missing columns : {sorted(missing)}")
        print(f"   Extra columns   : {sorted(extra)}")
        print(f"   → Dropping table so Cell 11 recreates with the new schema ...")
        spark.sql(f"DROP TABLE IF EXISTS {table}")
        print(f"✓  {table} dropped — run Cell 11 to recreate and populate.")
    else:
        print(f"✓  {table} already has the current schema — no migration needed.")
        print(f"   Columns present : {sorted(live_cols)}")
except Exception as e:
    if "TABLE_OR_VIEW_NOT_FOUND" in str(e) or "not found" in str(e).lower():
        print(f"  {table} does not exist yet — Cell 11 will create it.")
    else:
        raise

In [ ]:
# ── Cell 11: Business Model Agent ─────────────────────────────────────────────
# WORKSTREAMS path (databricks/agents/workstreams) is already on sys.path from Cell 1.
import importlib, json

import business_model_agent as bma
importlib.reload(bma)
bma_result = bma.main()

print("\n=== Business Model Agent — Reasoning Trace ===")
import json
for step in bma_result["reasoning_trace"]:
    print(f"  Step {step['step']}: [{step['tool']}]")
    print(f"    Input  : {step['input']}")
    print(f"    Output : {step['output']}")
    if step['sources']:
        print(f"    Sources: {step['sources']}")
    print()

print("=== Revenue Model Flags ===")
for flag in bma_result["flags"]:
    emoji = {"Red": "🔴", "Yellow": "🟡", "Green": "🟢"}.get(flag["severity"], "⚪")
    print(f"  {emoji} [{flag['severity']}] {flag['metric']}: {flag['value']}")
    print(f"    Threshold: {flag['threshold']}")
    print(f"    Note: {flag['note']}")
    print(f"    Source: {flag['source_doc']}  confidence={flag['confidence']}")
    print()

if bma_result.get("overlay_conflict"):
    print(f"⚠ Overlay conflict: {bma_result['overlay_conflict_note']}")

print("=== Data Room Gaps ===")
for gap in bma_result.get("data_room_gaps", []):
    print(f"  ! {gap}")

In [ ]:
# ── Cell 11b: Inspect Business Model Agent — new rich fields ──────────────────
# Run after Cell 11. Inspects the structured output introduced in the June 2026
# rewrite: products/services with per-offering margins, sales motion, revenue
# visibility, key dependencies, recent model changes, and overlay-specific
# customer fields (referral breakdown + payor mix for healthcare).
import json

print(f"CIM detected     : {bma_result.get('cim_detected')}")
print(f"Revenue model tag: {bma_result.get('revenue_model_tag')}")
print(f"Revenue model pct: {bma_result.get('revenue_model_pct_split')}")
print(f"Revenue model note: {bma_result.get('revenue_model_note')}")
print()

# ── Products & Services ───────────────────────────────────────────────────────
print("=== Products & Services ===")
for p in json.loads(bma_result.get("products_services_json") or "[]"):
    print(f"  {p.get('name')}")
    print(f"    gm_pct_stated  : {p.get('gm_pct_stated')}  ({p.get('gm_pct_note')})")
    print(f"    avg_price/rate : {p.get('avg_price_or_rate')}")
    print(f"    revenue_pct    : {p.get('revenue_pct')}")
    print(f"    source         : {p.get('source_doc')} — {p.get('source_location')}")
    print()

# ── Sales Motion ──────────────────────────────────────────────────────────────
print("=== Sales Motion ===")
sm = json.loads(bma_result.get("sales_motion_json") or "{}")
print(f"  tag             : {sm.get('tag')}")
print(f"  description     : {sm.get('description')}")
print(f"  key_roles       : {sm.get('key_roles')}")
print(f"  process_note    : {sm.get('process_note')}")
print(f"  compensation    : {sm.get('compensation_model')}")
print(f"  source          : {sm.get('source_doc')} — {sm.get('source_location')}")
print()

# ── Revenue Visibility ────────────────────────────────────────────────────────
print("=== Revenue Visibility ===")
rv = json.loads(bma_result.get("revenue_visibility_json") or "{}")
for k, v in rv.items():
    if v:
        print(f"  {k}: {v}")
print()

# ── Key Dependencies ──────────────────────────────────────────────────────────
print("=== Key Dependencies ===")
kd_list = json.loads(bma_result.get("key_dependencies_json") or "[]")
if kd_list:
    for kd in kd_list:
        risk_flag = " ⚠" if str(kd.get("concentration_risk")).lower() == "true" else ""
        print(f"  [{kd.get('dependency_type')}] {kd.get('name')}{risk_flag}")
        print(f"    {kd.get('description')}")
        print(f"    source: {kd.get('source_doc')}")
else:
    print("  (none extracted — check data_room_gaps)")
print()

# ── Recent Model Changes ──────────────────────────────────────────────────────
print("=== Recent Model Changes ===")
rc_list = json.loads(bma_result.get("recent_model_changes_json") or "[]")
if rc_list:
    for rc in rc_list:
        print(f"  [{rc.get('change_type')}] {rc.get('approximate_date')}: {rc.get('description')}")
        if rc.get("impact_note"):
            print(f"    impact: {rc.get('impact_note')}")
else:
    print("  (none extracted — check data_room_gaps)")
print()

# ── Customer Profile ──────────────────────────────────────────────────────────
print("=== Customer Profile ===")
cp = json.loads(bma_result.get("customer_profile_json") or "{}")
print(f"  segments_description  : {cp.get('segments_description')}")
print(f"  end_markets           : {cp.get('end_markets')}")
print(f"  geographic_concentration: {cp.get('geographic_concentration')}")

tenure = cp.get("client_tenure") or {}
if any(tenure.values()):
    print(f"  client_tenure         : {tenure.get('avg_tenure_stated')}  ({tenure.get('tenure_distribution_note')})")

ov_specific = cp.get("overlay_specific") or {}

# Healthcare sub-block
hc = ov_specific.get("healthcare") or {}
refs = hc.get("referral_source_breakdown") or []
payors = hc.get("payor_mix") or []
if refs or payors:
    print(f"\n  -- Healthcare overlay --")
    print(f"  referral_source_breakdown ({len(refs)} items):")
    for ref in refs:
        print(f"    {ref.get('source_type')}: {ref.get('pct_of_volume_or_profit')}  period={ref.get('period')}")
    print(f"  payor_mix ({len(payors)} items):")
    for p in payors:
        print(f"    {p.get('payor_type')}: {p.get('pct_of_revenue')}")

# Tech services sub-block
tech = ov_specific.get("tech_services") or {}
if tech.get("customer_size_mix") or tech.get("vertical_concentration"):
    print(f"\n  -- Tech services overlay --")
    print(f"  customer_size_mix     : {tech.get('customer_size_mix')}")
    print(f"  vertical_concentration: {tech.get('vertical_concentration')}")

# SaaS sub-block
saas = ov_specific.get("b2b_saas") or {}
if saas.get("arr_by_tier") or saas.get("icp_description"):
    print(f"\n  -- B2B SaaS overlay --")
    print(f"  arr_by_tier    : {saas.get('arr_by_tier')}")
    print(f"  icp_description: {saas.get('icp_description')}")

print()

# ── Executive summary + report path ──────────────────────────────────────────
print("=== Executive Summary ===")
print(f"  {bma_result.get('executive_summary')}")
print()
print(f"✓ Report → {bma_result.get('report_path')}")

# ── Source doc validation spot-check ─────────────────────────────────────────
print("\n=== Source doc validation (spot-check for 'COMPANY PROFILE' citations) ===")
citations = json.loads(bma_result.get("citations") or "[]")
bad_cits = [c for c in citations if "COMPANY PROFILE" in (c.get("document") or "").upper()]
if bad_cits:
    print(f"  ⚠  {len(bad_cits)} citation(s) still reference COMPANY PROFILE:")
    for c in bad_cits:
        print(f"    field={c.get('field')}  doc={c.get('document')}")
else:
    print(f"  ✓  No citations reference COMPANY PROFILE ({len(citations)} total citations checked)")

In [ ]:
# ── Cell 12: Financial Trends Agent ───────────────────────────────────────────
# WORKSTREAMS path is already on sys.path from Cell 1.
import importlib, json
import financial_trends_agent as fta
importlib.reload(fta)
fta_result = fta.main()

print("\n=== Financial Trends Agent — Reasoning Trace ===")
for step in fta_result["reasoning_trace"]:
    print(f"  Step {step['step']}: [{step['tool']}]")
    print(f"    Input  : {step['input']}")
    print(f"    Output : {step['output']}")
    print()

print("=== Financial Flags ===")
for flag in fta_result["flags"]:
    emoji = {"Red": "🔴", "Yellow": "🟡", "Green": "🟢"}.get(flag["severity"], "⚪")
    print(f"  {emoji} [{flag['severity']}] {flag['metric']}: {flag['value']}")
    print(f"    Threshold : {flag['threshold']}")
    print(f"    Note      : {flag['note']}")
    print(f"    Source    : {flag['source_doc']}  confidence={flag['confidence']}")
    print()

print("=== Addback Summary ===")
addbacks = json.loads(fta_result.get("addback_schedule_json") or "[]")
print(f"  {len(addbacks)} addback items found")
print(f"  Addback % of EBITDA: {fta_result.get('addback_pct_of_ebitda')}")
if addbacks:
    for a in addbacks[:5]:
        print(f"    - {a.get('description')}: {a.get('amount_stated')}  "
              f"support_doc={a.get('supporting_doc_referenced')}")

print("\n=== Data Room Gaps ===")
for gap in fta_result.get("data_room_gaps", []):
    print(f"  ! {gap}")

print("\n=== Discrepancies Found ===")
discrepancies = json.loads(fta_result.get("discrepancies") or "[]")
for d in discrepancies:
    print(f"  ⚠ {d['metric']}: {d['conflicting_values']}")
    print(f"    {d['note']}")

In [ ]:
# ── Cell 12b: Financial Story Assessment ──────────────────────────────────────
# Requires fta_result from Cell 12.  Calls generate_financial_assessment(), which:
#   1. Builds seven deterministic tables (revenue trend, gross margin, EBITDA
#      multi-version, segment, addbacks, budget-vs-actual, working capital).
#   2. Makes one LLM call to write 12-section narrative (improving / deteriorating /
#      distorted by mix / addbacks / timing).
#   3. Writes the final markdown to
#      /Volumes/{catalog}/analysis/reports/{company}/financial_assessment.md
#
# The cell renders the result inline as HTML when the `markdown` package is
# available, and falls back to printing raw markdown otherwise.

llm_endpoint = dbutils.widgets.get("llm_endpoint")
catalog      = dbutils.widgets.get("catalog")

assessment_md = fta.generate_financial_assessment(
    result=fta_result,
    spark=spark,
    llm_endpoint=llm_endpoint,
    catalog=catalog,
    write_to_volume=True,
)

# ── Render inline ──────────────────────────────────────────────────────────────
try:
    import markdown as _md_lib

    _html_body = _md_lib.markdown(
        assessment_md,
        extensions=["tables", "nl2br"],
    )
    displayHTML(f"""
    <div style="font-family: Georgia, 'Times New Roman', serif;
                max-width: 960px; margin: 24px auto; line-height: 1.65;
                color: #1a1a1a; padding: 0 16px;">
        <style>
            h1 {{ font-size: 1.8em; border-bottom: 2px solid #444; padding-bottom: 6px; }}
            h2 {{ font-size: 1.35em; margin-top: 2em; border-bottom: 1px solid #ccc; padding-bottom: 4px; }}
            h3 {{ font-size: 1.1em; margin-top: 1.6em; color: #333; }}
            table {{ border-collapse: collapse; width: 100%; margin: 1em 0; font-size: 0.9em; }}
            th {{ background: #f0f0f0; text-align: left; padding: 6px 10px; border: 1px solid #ccc; }}
            td {{ padding: 5px 10px; border: 1px solid #ddd; vertical-align: top; }}
            tr:nth-child(even) td {{ background: #fafafa; }}
            blockquote {{ background: #f9f9e8; border-left: 4px solid #c8a800;
                          margin: 1em 0; padding: 8px 16px; font-style: italic; }}
            code {{ background: #f4f4f4; padding: 1px 4px; border-radius: 3px; }}
            hr {{ border: none; border-top: 1px solid #ddd; margin: 2em 0; }}
        </style>
        {_html_body}
    </div>
    """)

except ImportError:
    # markdown package not installed — print raw text (still fully readable)
    print(assessment_md)

In [ ]:
# ── Cell 13: Verify Phase 3 analysis tables (Business Model + Financial Trends) ─
import json
company_name = dbutils.widgets.get("sp_company_name")

display(spark.sql(f"""
    SELECT company_name, cim_detected, revenue_model_tag,
           revenue_durability_flag, flag_confidence,
           sales_motion_tag, overlay_conflict, created_at
    FROM uc13.analysis.business_model
    WHERE company_name = '{company_name}'
    ORDER BY created_at DESC LIMIT 3
"""))

display(spark.sql(f"""
    SELECT company_name, industry_overlay_used,
           addback_pct_of_ebitda,
           SIZE(data_room_gaps) AS gap_count,
           created_at
    FROM uc13.analysis.financial_trends
    WHERE company_name = '{company_name}'
    ORDER BY created_at DESC LIMIT 3
"""))

row = spark.sql(f"""
    SELECT flags FROM uc13.analysis.financial_trends
    WHERE company_name = '{company_name}'
    ORDER BY created_at DESC LIMIT 1
""").collect()
if row:
    flags = json.loads(row[0]["flags"] or "[]")
    print(f"\nFinancial flags ({len(flags)} total):")
    for f in flags:
        emoji = {"Red": "🔴", "Yellow": "🟡", "Green": "🟢"}.get(f["severity"], "⚪")
        print(f"  {emoji} {f['metric']}: {f['value']} (threshold: {f['threshold']})")

---
## Phase 3 — Workstream Agents (Customer Quality · KPI · Legal · QofE)

Run each cell independently. **Dependency order:**
- Cells 14–15 can run in parallel (no inter-agent dependency).
- Cell 16 (Legal) should run **after** Cell 14 (Customer Quality) — it reads `contract_trigger_list`.
- Cell 17 (QofE) should run **after** Cell 12 (Financial Trends) — it reads `addback_schedule_json`.
- Cell 18 queries all six `uc13.analysis.*` tables for a final status view.

In [ ]:
# ── Cell 14: Customer Quality Agent ───────────────────────────────────────────
# Extracts customer concentration, NRR/GRR, payor mix.
# Generates contract_trigger_list for customers >20% revenue.
import importlib, json

import customer_quality_agent as cqa
importlib.reload(cqa)
cqa_result = cqa.main()

print("\n=== Customer Quality Agent — Reasoning Trace ===")
for step in cqa_result["reasoning_trace"]:
    print(f"  Step {step['step']}: [{step['tool']}]")
    print(f"    Input  : {step['input']}")
    print(f"    Output : {step['output']}")
    print()

print("=== Customer Quality Flags ===")
for flag in cqa_result["flags"]:
    emoji = {"Red": "🔴", "Yellow": "🟡", "Green": "🟢"}.get(flag["severity"], "⚪")
    print(f"  {emoji} [{flag['severity']}] {flag['metric']}: {flag['value']}")
    print(f"    Threshold : {flag['threshold']}")
    print(f"    Note      : {flag['note']}")
    print(f"    Source    : {flag['source_doc']}  confidence={flag['confidence']}")
    print()

print("=== Contract Trigger List ===")
triggers = [json.loads(t) for t in cqa_result.get("contract_trigger_list") or []]
if triggers:
    for t in triggers:
        print(f"  ▶ {t['customer_name']}  ({t['revenue_pct']}%)  vdr={t.get('contract_found_in_vdr')}")
else:
    print("  (no customers above 20% threshold)")

print("\n=== Data Room Gaps ===")
for gap in cqa_result.get("data_room_gaps", []):
    print(f"  ! {gap}")

print(f"\n✓ Executive Summary: {cqa_result.get('executive_summary')}")
print(f"✓ Report → {cqa_result.get('report_path')}")

In [ ]:
# ── Cell 15: KPI Agent ────────────────────────────────────────────────────────
# Extracts overlay-specific KPIs (tech utilization / healthcare census & turnover).
# Missing expected KPIs are returned as management questions (data_room_gaps).
import importlib, json

import kpi_agent as kpi
importlib.reload(kpi)
kpi_result = kpi.main()

print("\n=== KPI Agent — Reasoning Trace ===")
for step in kpi_result["reasoning_trace"]:
    print(f"  Step {step['step']}: [{step['tool']}]")
    print(f"    Input  : {step['input']}")
    print(f"    Output : {step['output']}")
    print()

print(f"=== KPI Flags (overlay: {kpi_result.get('overlay_confirmed')}) ===")
for flag in kpi_result["flags"]:
    emoji = {"Red": "🔴", "Yellow": "🟡", "Green": "🟢"}.get(flag["severity"], "⚪")
    print(f"  {emoji} [{flag['severity']}] {flag['metric']}: {flag['value']}")
    print(f"    Threshold : {flag['threshold']}")
    print(f"    Note      : {flag['note']}")
    print()

print("=== Missing KPIs (management questions) ===")
missing = json.loads(kpi_result.get("missing_kpis_json") or "[]")
for m in missing:
    print(f"  ? [{m.get('overlay')}] {m.get('kpi_name')}: {m.get('management_question')}")

print("\n=== Data Room Gaps ===")
for gap in kpi_result.get("data_room_gaps", []):
    print(f"  ! {gap}")

print(f"\n✓ Executive Summary: {kpi_result.get('executive_summary')}")
print(f"✓ Report → {kpi_result.get('report_path')}")

In [ ]:
# ── Cell 16: Legal Contracts Agent ────────────────────────────────────────────
# Reads contract_trigger_list from uc13.analysis.customer_quality (run Cell 14 first).
# Extracts contract register (CoC, termination, restrictive covenants) + litigation.
import importlib, json

import legal_contracts_agent as lca
importlib.reload(lca)
lca_result = lca.main()

print("\n=== Legal Contracts Agent — Reasoning Trace ===")
for step in lca_result["reasoning_trace"]:
    print(f"  Step {step['step']}: [{step['tool']}]")
    print(f"    Input  : {step['input']}")
    print(f"    Output : {step['output']}")
    print()

print(f"=== Legal Flags  ({lca_result.get('triggered_reviews_loaded')} triggered reviews loaded) ===")
for flag in lca_result["flags"]:
    emoji = {"Red": "🔴", "Yellow": "🟡", "Green": "🟢"}.get(flag["severity"], "⚪")
    print(f"  {emoji} [{flag['severity']}] {flag['metric']}: {flag['value']}")
    print(f"    Note   : {flag['note']}")
    print()

print("=== CoC Consent List ===")
coc_list = json.loads(lca_result.get("coc_consent_list_json") or "[]")
if coc_list:
    for c in coc_list:
        print(f"  ⚠ Contract {c.get('contract_id')}: {c.get('counterparty_name')}  rev={c.get('revenue_pct')}%")
        print(f"    Standard: {c.get('consent_standard_note')}")
else:
    print("  (no contracts requiring CoC consent found)")

print("\n=== Termination Exposure (<90 days notice) ===")
tfe = json.loads(lca_result.get("termination_exposure_json") or "[]")
for t in tfe:
    print(f"  ⚠ {t.get('counterparty_name')}: {t.get('notice_days')} days  rev={t.get('revenue_pct')}%")

print("\n=== Data Room Gaps ===")
for gap in lca_result.get("data_room_gaps", []):
    print(f"  ! {gap}")

print(f"\n✓ Executive Summary: {lca_result.get('executive_summary')}")
print(f"✓ Report → {lca_result.get('report_path')}")

In [ ]:
# ── Cell 17: Quality of Earnings Agent ────────────────────────────────────────
# Reads addback_schedule_json from uc13.analysis.financial_trends (run Cell 12 first).
# Classifies addbacks (Tier 1–4), detects revenue quality flags, computes EBITDA scenarios.
import importlib, json

import quality_of_earnings_agent as qoe
importlib.reload(qoe)
qoe_result = qoe.main()

print("\n=== Quality of Earnings Agent — Reasoning Trace ===")
for step in qoe_result["reasoning_trace"]:
    print(f"  Step {step['step']}: [{step['tool']}]")
    print(f"    Input  : {step['input']}")
    print(f"    Output : {step['output']}")
    print()

print("=== QofE Flags ===")
for flag in qoe_result["flags"]:
    emoji = {"Red": "🔴", "Yellow": "🟡", "Green": "🟢"}.get(flag["severity"], "⚪")
    print(f"  {emoji} [{flag['severity']}] {flag['metric']}: {flag['value']}")
    print(f"    Note   : {flag['note']}")
    print()

print("=== Addback Summary ===")
ledger = json.loads(qoe_result.get("addback_ledger_json") or "[]")
by_tier = {}
for a in ledger:
    t = a.get("tier_classification", "Unknown")
    by_tier.setdefault(t, []).append(a)
for tier, items in sorted(by_tier.items()):
    print(f"  {tier}: {len(items)} item(s)")
    for a in items[:3]:
        print(f"    - {a.get('description')[:60]}: {a.get('amount_dollars')}")

print("\n=== EBITDA Scenarios ===")
scenarios = json.loads(qoe_result.get("ebitda_scenarios_json") or "{}")
for k, v in scenarios.items():
    if k != "note":
        print(f"  {k}: {v}")

print(f"\n  Tier 4 count              : {qoe_result.get('tier4_addback_count')}")
print(f"  Total addbacks % of EBITDA: {qoe_result.get('total_addbacks_pct_of_ebitda')}")
print(f"  QofE report in VDR        : {qoe_result.get('qofe_report_present')}")

print("\n=== Data Room Gaps ===")
for gap in qoe_result.get("data_room_gaps", []):
    print(f"  ! {gap}")

print(f"\n✓ Executive Summary: {qoe_result.get('executive_summary')}")
print(f"✓ Report → {qoe_result.get('report_path')}")

In [ ]:
# ── Cell 18: Verify all six analysis tables ───────────────────────────────────
import json
company_name = dbutils.widgets.get("sp_company_name")

analysis_tables = {
    "business_model":      "uc13.analysis.business_model",
    "financial_trends":    "uc13.analysis.financial_trends",
    "customer_quality":    "uc13.analysis.customer_quality",
    "kpi":                 "uc13.analysis.kpi",
    "legal_contracts":     "uc13.analysis.legal_contracts",
    "quality_of_earnings": "uc13.analysis.quality_of_earnings",
}

print(f"{'Agent':<22} {'Rows':>5}  {'Flags':>6}  {'Gaps':>5}  {'Report path'}")
print("-" * 90)
for label, fqt in analysis_tables.items():
    try:
        rows = spark.sql(f"""
            SELECT flags, data_room_gaps, report_path
            FROM {fqt}
            WHERE company_name = '{company_name}'
            ORDER BY created_at DESC LIMIT 1
        """).collect()
        if rows:
            r = rows[0]
            flag_count = len(json.loads(r["flags"] or "[]"))
            gap_count  = len(r["data_room_gaps"] or [])
            rpt        = (r["report_path"] or "—")[-55:]
            print(f"  {label:<20} {'1':>5}  {flag_count:>6}  {gap_count:>5}  ...{rpt}")
        else:
            print(f"  {label:<20} {'0':>5}  {'—':>6}  {'—':>5}  (not run yet)")
    except Exception as e:
        print(f"  {label:<20} {'ERR':>5}  —  {str(e)[:50]}")

print()

# Detailed flag summary across all agents
for label, fqt in analysis_tables.items():
    try:
        row = spark.sql(f"""
            SELECT flags FROM {fqt}
            WHERE company_name = '{company_name}'
            ORDER BY created_at DESC LIMIT 1
        """).collect()
        if row:
            flags = json.loads(row[0]["flags"] or "[]")
            reds    = [f for f in flags if f["severity"] == "Red"]
            yellows = [f for f in flags if f["severity"] == "Yellow"]
            if reds or yellows:
                print(f"\n── {label} ({'🔴' * len(reds)}{'🟡' * len(yellows)}) ──")
                for f in reds + yellows:
                    emoji = "🔴" if f["severity"] == "Red" else "🟡"
                    print(f"  {emoji} {f['metric']}: {f['value']}")
    except Exception:
        pass